# 🚀 TripMate AI: Sistem Rekomendasi Destinasi Wisata
**Metode:** Content-Based Filtering menggunakan TF-IDF & Cosine Similarity  

Notebook ini berisi alur pembuatan model *Machine Learning* untuk sistem TripMate AI. Model ini bertugas memberikan rekomendasi tempat wisata yang dipersonalisasi berdasarkan preferensi kategori, lokasi (provinsi/kota), dan batas anggaran (budget) dari pengguna.

---
## TAHAP 1: Import Library & Load Dataset
Langkah pertama adalah memanggil library yang dibutuhkan (Pandas) dan memuat dataset wisata yang sudah dibersihkan sebelumnya.

In [1]:
import pandas as pd

# 1. Load dataset baru yang sudah dirapikan
df = pd.read_csv('wisata_indonesia_super_cleanss.csv')

print("=== INFO DATASET TRIPMATE AI ===")
print(f"Total Data: {len(df)} destinasi wisata.")
print(f"Kolom yang tersedia: {list(df.columns)}")
print("-" * 60)

print("\n=== PANDUAN INPUT KATEGORI VALID & JUMLAHNYA ===")
# Menampilkan 6 kategori hasil penyederhanaan
daftar_kategori = df['kategori'].value_counts()
for kat, jumlah in daftar_kategori.items():
    print(f"- {kat:<30} : {jumlah} tempat wisata")
print("-" * 60)

print("\n=== DAFTAR 10 PROVINSI TERBANYAK DI DATASET ===")
daftar_provinsi = df['provinsi'].value_counts().head(100)
for prov, jumlah in daftar_provinsi.items():
    print(f"- {prov:<25} : {jumlah} tempat wisata")

print("\n=== DAFTAR 10 Kabupaten TERBANYAK DI DATASET ===")
daftar_provinsi = df['kota_kabupaten'].value_counts().head(100)
for prov, jumlah in daftar_provinsi.items():
    print(f"- {prov:<25} : {jumlah} tempat wisata")

print("\n=== STATISTIK HARGA TIKET MASUK (PRICE) ===")
print(f"- Harga Tiket Termurah : Rp {df['price'].min():,}")
print(f"- Harga Tiket Termahal : Rp {df['price'].max():,}")
print(f"- Rata-rata Harga Tiket: Rp {int(df['price'].mean()):,}")

=== INFO DATASET TRIPMATE AI ===
Total Data: 1025 destinasi wisata.
Kolom yang tersedia: ['kategori', 'nama_wisata', 'price', 'latitude', 'longitude', 'alamat', 'provinsi', 'kota_kabupaten', 'deskripsi_bersih', 'Image_Path']
------------------------------------------------------------

=== PANDUAN INPUT KATEGORI VALID & JUMLAHNYA ===
- Cagar Alam & Wisata Alam       : 327 tempat wisata
- Budaya & Sejarah               : 213 tempat wisata
- Taman Hiburan & Rekreasi       : 185 tempat wisata
- Pusat Perbelanjaan & Kuliner   : 147 tempat wisata
- Bahari (Pantai)                : 104 tempat wisata
- Tempat Ibadah & Religi         : 49 tempat wisata
------------------------------------------------------------

=== DAFTAR 10 PROVINSI TERBANYAK DI DATASET ===
- Jawa Timur                : 129 tempat wisata
- Jawa Barat                : 104 tempat wisata
- Jawa Tengah               : 99 tempat wisata
- Bali                      : 98 tempat wisata
- DKI Jakarta               : 81 tempat wisata


## TAHAP 2: Eksplorasi Data Awal (Data Understanding)
Pada tahap ini, kita akan melihat sekilas bentuk tabel dari dataset yang digunakan beserta informasi tipe datanya. Ini untuk memastikan bahwa semua kolom yang dibutuhkan oleh AI (seperti kategori, deskripsi, lokasi, dan harga tiket) sudah terbaca dengan benar oleh sistem.

In [2]:
df

,kategori,nama_wisata,price,latitude,longitude,alamat,provinsi,kota_kabupaten,deskripsi_bersih,Image_Path
0,Cagar Alam & Wisata Alam,Air Terjun Aling-Aling,10000,-8.176792,115.106284,"Air Terjun Aling-Aling, Panji Anom, Buleleng, ...",Bali,Buleleng,Air Terjun Aling-Aling merupakan salah satu de...,https://ksmtour.com/media/images/articles6/air...
1,Cagar Alam & Wisata Alam,Air Terjun Bantimurung,10000,-5.016520,119.685520,"Air Terjun Bantimurung, Jalan Poros Bantimurun...",Sulawesi Selatan,Sulawesi,Air Terjun Bantimurung adalah salah satu air t...,https://cdn-2.tstatic.net/makassar/foto/bank/i...
2,Cagar Alam & Wisata Alam,Air Terjun Benang Kelambu,10000,-8.532770,116.337011,"Air Terjun Benang Kelambu, Pemotoh, Lombok Ten...",Nusa Tenggara Barat,Lombok Tengah,Air Terjun Benang Kelambu merupakan salah satu...,https://cdn.idntimes.com/content-images/commun...
3,Cagar Alam & Wisata Alam,Air Terjun Benang Stokel,10000,-8.533036,116.341379,"Air Terjun Benang Stokel, Pemotoh, Lombok Teng...",Nusa Tenggara Barat,Lombok Tengah,Air Terjun Benang Stokel merupakan salah satu ...,https://s-light.tiket.photos/t/01E25EBZS3W0FY9...
4,Cagar Alam & Wisata Alam,Air Terjun Bidadari,10000,-2.928361,107.839043,"Air Terjun Bidadari, Nyuruk, Belitung Timur, K...",Kepulauan Bangka Belitung,Belitung Timur,Air Terjun Bidadari merupakan salah satu desti...,https://assets.pikiran-rakyat.com/crop/0x0:0x0...
...,...,...,...,...,...,...,...,...,...,...
1020,Cagar Alam & Wisata Alam,Situ Cileunca,2500,-7.202004,107.547142,"Situ Cileunca, Kampung Cibuluh, Pulosari, Kabu...",Jawa Barat,Kabupaten Bandung,"Situ Cileunca (Aksara Sunda Baku: ), adala...",https://ksmtour.com/media/images/articles17/si...
1021,Cagar Alam & Wisata Alam,Situ Cisanti,15000,-7.209391,107.657477,"Situ Cisanti, Kertasari, Kabupaten Bandung, Ja...",Jawa Barat,Kabupaten Bandung,Situ Cisanti (Aksara Sunda Baku: ) adalah d...,https://travelspromo.com/wp-content/uploads/20...
1022,Cagar Alam & Wisata Alam,Situ Patenggang,20000,-7.165260,107.369782,"Situ Patenggang, Bauan, Rancabali, Kabupaten B...",Jawa Barat,Kabupaten Bandung,Situ Patenggang atau Situ Patengan adalah suat...,https://ik.imagekit.io/tvlk/blog/2018/08/Bandu...
1023,Cagar Alam & Wisata Alam,Situ Rawa Gede,15000,-6.292928,106.977288,"Situ Rawa Gede, Pesona Metropolitan, Bojong Me...",Jawa Barat,Bekasi,Kabupaten Purwakarta (bahasa Sunda: ) adalah ...,https://assets.pikiran-rakyat.com/crop/0x0:0x0...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   kategori          1025 non-null   object 
 1   nama_wisata       1025 non-null   object 
 2   price             1025 non-null   int64  
 3   latitude          1025 non-null   float64
 4   longitude         1025 non-null   float64
 5   alamat            1025 non-null   object 
 6   provinsi          1025 non-null   object 
 7   kota_kabupaten    1025 non-null   object 
 8   deskripsi_bersih  1025 non-null   object 
 9   Image_Path        1025 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 80.2+ KB


## TAHAP 3: Pengecekan Kualitas Data (Missing Values)
Sebelum data diajarkan ke model AI, sangat penting untuk melakukan validasi kualitas data. Tahap ini bertujuan untuk mendeteksi apakah ada nilai yang kosong (*missing values*) di dalam dataset. Jika ada data yang kosong, maka akan diisi atau ditangani agar tidak menyebabkan *error* pada saat perhitungan matematis AI.

In [4]:
# =====================================================================
# TAHAP 1: DETEKSI LOKASI MISSING VALUES
# =====================================================================
# Mencari kolom apa saja yang memiliki data kosong
missing_counts = df.isnull().sum()
kolom_missing = missing_counts[missing_counts > 0].index.tolist()

print("="*50)
print("1. LAPORAN MISSING VALUES")
print("="*50)

if not kolom_missing:
    print("Hebat! Dataset kamu sudah bersih, tidak ada missing values.")
else:
    for col in kolom_missing:
        print(f"• Kolom '{col}' memiliki {missing_counts[col]} data kosong.")

    print("\nBerikut adalah 5 contoh baris yang datanya bolong:")
    # Menampilkan baris yang memiliki minimal 1 missing value
    baris_bolong = df[df.isnull().any(axis=1)]

    # Kita tampilkan kolom 'nama_wisata' dan kolom yang bolong saja agar mudah dibaca
    kolom_tampilan = ['nama_wisata'] + kolom_missing
    print(baris_bolong[kolom_tampilan].head())




1. LAPORAN MISSING VALUES
Hebat! Dataset kamu sudah bersih, tidak ada missing values.


## TAHAP 4: Pembuatan Model AI (Content-Based Filtering)
Di sinilah inti kecerdasan buatan dibuat. Sistem akan melakukan beberapa hal:
1. **Metadata Soup:** Menggabungkan kolom Kategori, Provinsi, Kota, dan Deskripsi menjadi satu kalimat utuh. Kolom 'kategori' diberi bobot lebih besar (diulang 3x) agar AI memprioritaskan jenis wisata.
2. **TF-IDF Vectorization:** Mengubah teks bahasa manusia (deskripsi wisata) menjadi angka/vektor yang bisa dihitung oleh komputer, sekaligus membuang kata-kata hubung yang tidak penting (*stopwords*).

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Preprocessing dasar untuk memastikan tidak ada nilai kosong (NaN)
df['deskripsi_bersih'] = df['deskripsi_bersih'].fillna('')
df['kategori'] = df['kategori'].fillna('').astype(str)
df['provinsi'] = df['provinsi'].fillna('').astype(str)
df['kota_kabupaten'] = df['kota_kabupaten'].fillna('').astype(str)

# Membuat Metadata Soup (Kolom kategori baru diulang 3x agar bobotnya paling kuat)
df['metadata_soup'] = (df['kategori'] + ' ') * 3 + df['provinsi'] + ' ' + df['kota_kabupaten'] + ' ' + df['deskripsi_bersih']
df['metadata_soup'] = df['metadata_soup'].str.lower()

# Stopwords untuk mengabaikan kata hubung bawaan teks deskripsi
indonesian_stopwords = [
    'yang', 'di', 'dan', 'adalah', 'untuk', 'sebuah', 'ini', 'itu',
    'atau', 'pada', 'ke', 'dari', 'terdapat', 'menjadi', 'salah', 'satu', 'juga', 'ada'
]

# Inisialisasi TF-IDF dengan N-Gram (1,2) agar frasa seperti "wisata alam" terbaca utuh
tfidf = TfidfVectorizer(stop_words=indonesian_stopwords, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['metadata_soup'])

print("Model AI Content-Based Filtering berhasil dilatih menggunakan kategori baru!")

Model AI Content-Based Filtering berhasil dilatih menggunakan kategori baru!


## TAHAP 5: Fungsi Mesin Rekomendasi (API Ready)
Fungsi ini adalah mesin utama yang akan dipanggil oleh *Backend* (sistem web). Alur kerjanya:
1. Menyaring data secara mutlak berdasarkan input **Lokasi** dan **Budget Maksimal**.
2. Membaca input **Kategori** dari pengguna (bisa satu tombol atau banyak tombol sekaligus).
3. Menggunakan **Cosine Similarity** untuk mengukur seberapa mirip preferensi pengguna dengan deskripsi tempat wisata yang tersisa.
4. Mengubah hasil perhitungan menjadi format **JSON/Dictionary** agar mudah dikirim melalui REST API ke aplikasi web atau mobile.

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def get_trip_recommendations(user_categories, user_province_input=None, max_budget=None, top_n=5):
    """
    Fungsi untuk API Web.
    - user_categories: Bisa menerima teks biasa atau LIST (banyak tombol yang diklik).
    - max_budget: Menerima angka maksimal dari tombol slider harga.
    """
    mask = pd.Series(True, index=df.index)

    # 1. Filter Lokasi (Jika pengguna memilih dropdown/tombol provinsi)
    if user_province_input:
        location_clean = user_province_input.lower().strip()
        mask = mask & (
            df['provinsi'].str.lower().str.contains(location_clean, na=False) |
            df['kota_kabupaten'].str.lower().str.contains(location_clean, na=False)
        )

    # 2. Filter Harga Mutlak (Dari klik tombol harga)
    if max_budget is not None:
        mask = mask & (df['price'] <= max_budget)

    matched_indices = df[mask].index
    if len(matched_indices) == 0:
        return {"error": "Maaf, tidak ada destinasi yang cocok dengan kombinasi filter tersebut."}

    df_filtered = df.loc[matched_indices].reset_index(drop=True)
    matrix_filtered = tfidf_matrix[matched_indices]

    # 3. PENYESUAIAN UNTUK TOMBOL: Cek apakah input berupa list (beberapa tombol diklik)
    if isinstance(user_categories, list):
        # Gabungkan teks dari banyak tombol menjadi satu kalimat
        # Contoh: ["Budaya & Sejarah", "Bahari"] menjadi "budaya & sejarah bahari"
        user_input_cleaned = " ".join(user_categories).lower()
    else:
        # Jika cuma 1 tombol atau ketik manual
        user_input_cleaned = str(user_categories).lower()

    # Ubah teks gabungan tadi menjadi vektor AI
    user_vector = tfidf.transform([user_input_cleaned])

    # 4. Hitung Cosine Similarity untuk ranking
    similarity_scores = cosine_similarity(user_vector, matrix_filtered).flatten()
    top_indices = similarity_scores.argsort()[::-1][:top_n]

    recommendations = df_filtered.iloc[top_indices].copy()
    recommendations['score_akurasi'] = similarity_scores[top_indices]

    output_cols = ['nama_wisata', 'kategori', 'price', 'provinsi', 'kota_kabupaten', 'score_akurasi']
    hasil_akhir = recommendations[output_cols]

    # Output JSON untuk backend web
    return hasil_akhir.to_dict(orient='records')

## TAHAP 6: Simulasi Pengujian Sistem (Testing)
Tahap terakhir ini digunakan untuk menyimulasikan interaksi pengguna di aplikasi. Kita akan meniru aksi pengguna yang mencentang beberapa tombol kategori wisata, memilih lokasi, dan mengatur batas harga tiket, lalu melihat apakah AI mampu memberikan rekomendasi (dalam format JSON) dengan akurat.

In [9]:
import json

# Skenario: Pengguna mencentang tombol "Budaya & Sejarah" dan "Taman Hiburan & Rekreasi"
tombol_kategori_yang_diklik = ["Budaya & Sejarah", "pantai"]
tombol_provinsi = "Daerah Khusus ibukota Jakarta"
batas_harga = 10000

hasil = get_trip_recommendations(
    user_categories=tombol_kategori_yang_diklik,
    user_province_input=tombol_provinsi,
    max_budget=batas_harga,
    top_n=3
)

print(json.dumps(hasil, indent=4))

[
    {
        "nama_wisata": "Museum Bank Indonesia",
        "kategori": "Budaya & Sejarah",
        "price": 2000,
        "provinsi": "DKI Jakarta",
        "kota_kabupaten": "Daerah Khusus ibukota Jakarta",
        "score_akurasi": 0.35566324828936224
    },
    {
        "nama_wisata": "Museum Satria Mandala",
        "kategori": "Budaya & Sejarah",
        "price": 5000,
        "provinsi": "DKI Jakarta",
        "kota_kabupaten": "Daerah Khusus ibukota Jakarta",
        "score_akurasi": 0.33706952979284155
    },
    {
        "nama_wisata": "Museum Fatahillah",
        "kategori": "Budaya & Sejarah",
        "price": 5000,
        "provinsi": "DKI Jakarta",
        "kota_kabupaten": "Daerah Khusus ibukota Jakarta",
        "score_akurasi": 0.32233184733062187
    }
]


In [10]:
import joblib

# Simpan dataframe yang sudah bersih, model tfidf, dan matrix-nya
joblib.dump(df, 'travel_df.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(tfidf_matrix, 'tfidf_matrix.pkl')

['tfidf_matrix.pkl']